# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [ ]:
import pandas as pd
import sqlite3
from loguru import logger

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_V3_DWH.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

2026-03-29 13:02:33.974 | INFO     | __main__:<module>:10 - SDM en DWH connecties succesvol verbonden.


Dim_Klant (SCD TYPE 2) ROBBERT

In [5]:
# Wat willen we bereiken?
# We willen Dim_Klant in het DWH vullen met deze logica:
# - nieuwe klant in SDM, nog niet in DWH -> INSERT
# - bestaande klant, maar gewijzigd -> oude actuele rij afsluiten + nieuwe INSERT
# - bestaande klant, niet gewijzigd -> niets doen.

# business key = klantr uit SDM
# surrogate key = klant_sk in DWH

# We halen eerst brondata op uit beide klanttabellen.
# Dit doen we omdat Dim_Klant niet uit één bron komt, maar uit de tabellen:
# - Accessoire_Verkoop_Klant
# - Fiets_Verkoop_Klant
# We voegen deze dus daarom ook verticaal samen.

query_accessoire_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Accessoire_Verkoop_Klant
"""

query_fiets_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Fiets_Verkoop_Klant
"""
df_accessoire_klant = pd.read_sql_query(query_accessoire_klant, sdm_conn)
df_fiets_klant = pd.read_sql_query(query_fiets_klant, sdm_conn)

# We willen deze twee losse DataFrames verticaal samenvoegen.
# We maken dus één bronset voor Dim_Klant

df_klant_source = pd.concat([df_accessoire_klant, df_fiets_klant], ignore_index=True) # ignore_index true omdat we een nette nieuwe index willen.

# We verwijderen dubbele rijen, want we willen niet dezelfde klant twee keer in onze DWH hebben, voordat we hem vergelijken.
df_klant_source = df_klant_source.drop_duplicates().reset_index(drop=True)

print(df_klant_source.head())
print(f"Aantal unieke klanten in source: {len(df_klant_source)}")


   klantnr            naam           adres woonplaats geslacht geboortedatum
0        1      Jan Jansen   Kerkstraat 12  Amsterdam        M    1985-03-22
1        2  Sophie de Boer     Lindelaan 8  Rotterdam        V    1990-07-11
2        3   Pieter Visser   Havenstraat 3   Den Haag        M    1978-11-05
3        4       Emma Smit    Boomgaard 22    Haarlem        V    1995-02-18
4        5      Tom Bakker  Stationsweg 44     Leiden        M    1982-09-09
Aantal unieke klanten in source: 30


Dim_Accessoire (SCD TYPE 1) ROBBERT

Dim_Datum ROBBERT

Fct_Verkoop ROBBERT


Dim_Fiets (SCD Type 2) MEES

Dim_Monteur (SCD Type 1) MEES

Fct_Onderhoud MEES

Fct_Inkoop MEES